# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Simular un escenario real de ingesta de datos ruidosos (anomalías, nulos, duplicados), ejecutar un perfilamiento estadístico para dimensionar el caos, y procesarlos a través de un flujo ETL construyendo las capas lógicas de un Data Lake (Capa *Raw* y Capa *Curated*).



In [8]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

# Guardamos una base fresca para evitar el error de bytes corruptos en Colab
nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')

# a) Inyectamos un 5% de nulos
n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

# b) Duplicamos 50,000 registros
df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)

# c) Inyectamos outliers (valores absurdos que rompen la estadística)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

1/3 Generando datos base perfectos en memoria...
2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...
3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: 5,050,000


## Fase 1: Profiling y Capa RAW (El Diagnóstico del Caos)
Antes de transformar, debemos perfilar el archivo crudo para entender la magnitud del problema y luego guardarlo en la capa inmutable del Data Lake.

In [ ]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")

# 1. Auditoría de Nulos y Duplicados
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")

# 2. Auditoría de Outliers (Fíjense en la diferencia entre el promedio y el max)
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

# 3. Guardado en Capa RAW (Inmutable)
os.makedirs('./datalake/raw', exist_ok=True)
os.makedirs('./datalake/curated', exist_ok=True)
os.makedirs('./datalake/quarantine', exist_ok=True)  # Reto 1: nueva capa

print("\n--- EXTRACT & LOAD (EL) ---")
print("Guardando evidencia en la Capa RAW...")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')
print("¡Archivo RAW almacenado con éxito! Listos para limpiar.")

## Fase 2: Transform (T) - Pipeline ETL de Limpieza y Normalización
Primero se eliminan duplicados. **Reto 2:** las transacciones canceladas con monto nulo se descartan, mientras que los montos faltantes de otros estados se imputan mediante la mediana de su categoría. **Reto 1:** los valores atípicos identificados con IQR se guardan en la capa *Quarantine* en formato Parquet y se separan del conjunto limpio. Finalmente, se normaliza el monto para la capa *Curated*.

In [ ]:
print("Iniciando Transformación...\n")

# 1. Eliminación de Duplicados
antes = len(df_ingesta)
df_limpio = df_ingesta.drop_duplicates().copy()
print(f"Duplicados eliminados: {antes - len(df_limpio):,}")

# 2. Reto 2: Descarte inteligente e imputación por mediana
# Cancelado + monto nulo => descartar toda la fila
mascara_descarte = (df_limpio['estado'] == 'Cancelado') & (df_limpio['monto'].isna())
filas_descartadas = mascara_descarte.sum()
df_limpio = df_limpio.loc[~mascara_descarte].copy()
print(f"Transacciones Cancelado con monto nulo descartadas: {filas_descartadas:,}")

# Resto de estados: imputar nulos con la mediana de su categoría
medianas = df_limpio.groupby('categoria')['monto'].transform('median')
df_limpio['monto'] = df_limpio['monto'].fillna(medianas)
print(f"Nulos restantes en monto: {df_limpio['monto'].isnull().sum():,}")

# 3. Reto 1: Cuarentena de Outliers (Regla IQR)
Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

mascara_outliers = ((df_limpio['monto'] < limite_inferior) |
                    (df_limpio['monto'] > limite_superior))
df_cuarentena = df_limpio.loc[mascara_outliers].copy()

os.makedirs('./datalake/quarantine', exist_ok=True)
df_cuarentena.to_parquet('./datalake/quarantine/transacciones_outliers.parquet', engine='pyarrow', index=False)
print(f"Outliers enviados a QUARANTINE: {len(df_cuarentena):,}")

# Los outliers no se borran: quedan conservados en QUARANTINE.
# Solo se excluyen del conjunto que pasará a CURATED.
df_limpio = df_limpio.loc[~mascara_outliers].copy()

# 4. Normalización Min-Max (Escala de 0 a 1), como en el notebook base
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()
df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
print("Normalización Min-Max completada.")

## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el dato limpio en formato Parquet en *Curated*. Verificamos las tres carpetas: *Raw* conserva los datos originales, *Quarantine* guarda los valores atípicos para revisión y *Curated* contiene el conjunto limpio y normalizado.

In [ ]:
# 1. Guardamos en la capa analítica
df_limpio.to_parquet('./datalake/curated/transacciones_model_ready.parquet')
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 2. Demostración del dato limpio (El "Después")
print("--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---")
# Comparamos la estadística final para demostrar que la media volvió a la normalidad
print(df_limpio[['monto', 'monto_escalado']].describe().applymap(lambda x: f"{x:,.2f}"))

print("\n--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'monto', 'monto_escalado']].head(5))

print("\n--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---")
print(f"RAW (Sucio): {os.path.getsize('./datalake/raw/ingesta_transacciones_sucio.parquet') / (1024*1024):.2f} MB")
print(f"QUARANTINE (Outliers): {os.path.getsize('./datalake/quarantine/transacciones_outliers.parquet') / (1024*1024):.2f} MB")
print(f"CURATED (Limpio): {os.path.getsize('./datalake/curated/transacciones_model_ready.parquet') / (1024*1024):.2f} MB")

print("\n--- TRES CARPETAS DEL DATA LAKE ---")
for capa in ['raw', 'quarantine', 'curated']:
    carpeta = f'./datalake/{capa}'
    print(f"{carpeta}/ -> {os.listdir(carpeta)}")
print(f"\nRegistros en QUARANTINE: {len(df_cuarentena):,}")
print(f"Registros en CURATED: {len(df_limpio):,}")